<font size="+3">Duo Design CXCR4</font> 

In [2]:
import glob
import os
import shutil
import csv
import os
import sys
import json

import json
import numpy as np
import pandas as pd

from Bio.PDB import PDBParser, PPBuilder, MMCIFParser, PDBIO 
import MDAnalysis as mda
from MDAnalysis.analysis import align
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="MDAnalysis")

# Sequences

In [8]:
pdb_files = ['hCXCR4_af3_inactive.pdb', 'mCXCR4_af3_inactive.pdb']

for pdb_file in pdb_files:
    print(f"\n### {pdb_file}")
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure("prot", pdb_file)

    seqs = {}

    for model in structure:
        for chain in model:
            seq = []
            for residue in chain:
                if Polypeptide.is_aa(residue, standard=True):
                    seq.append(seq1(residue.resname))
            if seq:
                seqs[chain.id] = "".join(seq)

    for chain_id, sequence in seqs.items():
        print(f">chain_{chain_id}")
        print(sequence)



### hCXCR4_af3_inactive.pdb
>chain_A
FREENANFNKIFLPTIYSIIFLTGIVGNGLVILVMGYQKKLRSMTDKYRLHLSVADLLFVITLPFWAVDAVANWYFGNFLCKAVHVIYTVNLYSSVLILAFISLDRYLAIVHATNSQRPRKLLAEKVVYVGVWIPALLLTIPDFIFANVSEADDRYICDRFYPNDLWVVVFQFQHIMVGLILPGIVILSCYCIIISKLSHSKGHQKRKALKTTVILILAFFACWLPYYIGISIDSFILLEIIKQGCEFENTVHKWISITEALAFFHCCLNPILYAFLGAKFKTSAQHALTSVSRGSSLKILSKGKR

### mCXCR4_af3_inactive.pdb
>chain_A
FRDENVHFNRIFLPTIYFIIFLTGIVGNGLVILVMGYQKKLRSMTDKYRLHLSVADLLFVITLPFWAVDAMADWYFGKFLCKAVHIIYTVNLYSSVLILAFISLDRYLAIVHATNSQRPRKLLAEKAVYVGVWIPALLLTIPDFIFADVSQGDISQGDDRYICDRLYPDSLWMVVFQFQHIMVGLILPGIVILSCYCIIISKLSHSKGHQKRKALKTTVILILAFFACWLPYYVGISIDSFILLGVIKQGCDFESIVHKWISITEALAFFHCCLNPILYAFLGAKFKSSAQHALNSMSRGSSLKILSKGKR


# Sequence Identity

In [20]:
human = "FREENANFNKIFLPTIYSIIFLTGIVGNGLVILVMGYQKKLRSMTDKYRLHLSVADLLFVITLPFWAVDAVANWYFGNFLCKAVHVIYTVNLYSSVLILAFISLDRYLAIVHATNSQRPRKLLAEKVVYVGVWIPALLLTIPDFIFANVSEADDRYICDRFYPNDLWVVVFQFQHIMVGLILPGIVILSCYCIIISKLSHSKGHQKRKALKTTVILILAFFACWLPYYIGISIDSFILLEIIKQGCEFENTVHKWISITEALAFFHCCLNPILYAFLGAKFKTSAQHALTSVSRGSSLKILSKGKR"
mouse = "FRDENVHFNRIFLPTIYFIIFLTGIVGNGLVILVMGYQKKLRSMTDKYRLHLSVADLLFVITLPFWAVDAMADWYFGKFLCKAVHIIYTVNLYSSVLILAFISLDRYLAIVHATNSQRPRKLLAEKAVYVGVWIPALLLTIPDFIFADVSQGDISQGDDRYICDRLYPDSLWMVVFQFQHIMVGLILPGIVILSCYCIIISKLSHSKGHQKRKALKTTVILILAFFACWLPYYVGISIDSFILLGVIKQGCDFESIVHKWISITEALAFFHCCLNPILYAFLGAKFKSSAQHALNSMSRGSSLKILSKGKR"

# Perform global pairwise alignment
alignments = pairwise2.align.globalxx(human, mouse)

# Take the best alignment (highest score)
best_alignment = alignments[0]
aligned_human, aligned_mouse, score, start, end = best_alignment

# Calculate sequence identity
matches = sum(1 for a, b in zip(aligned_human, aligned_mouse) if a == b)
length = len(aligned_human)
seq_identity = (matches / length) * 100

print(f"Sequence Identity: {seq_identity:.2f}%")
print(f"Alignment length: {length}")

Sequence Identity: 83.63%
Alignment length: 336


# Remove Trajectories with Membrane Clashes

## Prepare References

In [4]:
def simple_rotation_matrix(a, b):
    a = a / np.linalg.norm(a)
    b = b / np.linalg.norm(b)
    v = np.cross(a, b)
    c = np.dot(a, b)
    K = np.array([
        [0, -v[2], v[1]],
        [v[2], 0, -v[0]],
        [-v[1], v[0], 0]
    ])
    return np.eye(3) + K + K @ K * (1 / (1 + c))

In [53]:
species = ['mouse', 'human']

for s in species:

    u = mda.Universe(f'{s[0]}CXCR4_short.pdb')
    sel = u.select_atoms('all')
    
    sel.translate(-sel.center_of_mass())
    
    # align principal axis with z
    principal_axis = sel.principal_axes()[2]
    principal_axis /= np.linalg.norm(principal_axis)

    if s == 'human':
        x = u.select_atoms("name CA and resid 167").center_of_mass()
        y = u.select_atoms("name CA and resid 178").center_of_mass()
    else:
        x = u.select_atoms("name CA and resid 172").center_of_mass()
        y = u.select_atoms("name CA and resid 183").center_of_mass()

    principal_axis = x-y
    principal_axis /= np.linalg.norm(principal_axis)

    z_axis = np.array([0, 0, 1])
    
    R = simple_rotation_matrix(principal_axis, z_axis)
    sel.rotate(R)
    
    sel.write(f'{s[0]}CXCR4_short.pdb')

## Align to References

In [ ]:
species = ['mouse', 'human']

outdir = 'Trajectory/Aligned'
os.makedirs(outdir, exist_ok=True)

for s in species:
    ref = mda.Universe(f'{s[0]}CXCR4_short.pdb')
    sel_ref = ref.select_atoms('chainID A')

    files = glob.glob(f'Trajectory/Relaxed/*_{s}.pdb')
    files.sort()

    for file in files:
        design = os.path.basename(file)
        u = mda.Universe(file) 
        sel_u = u.select_atoms('chainID A and not type H and not name OXT')

        align.alignto(u, ref, select='chainID A and not type H and not name OXT')
        sel = u.select_atoms('all')
        sel.write(f'{outdir}/{design}')


Trajectory/Relaxed/CXCR4_l100_s14635CXCR4_mouse.pdb
Trajectory/Relaxed/CXCR4_l100_s87186CXCR4_mouse.pdb
Trajectory/Relaxed/CXCR4_l101_s50501CXCR4_mouse.pdb
Trajectory/Relaxed/CXCR4_l102_s38852CXCR4_mouse.pdb
Trajectory/Relaxed/CXCR4_l103_s28173CXCR4_mouse.pdb
Trajectory/Relaxed/CXCR4_l103_s82425CXCR4_mouse.pdb
Trajectory/Relaxed/CXCR4_l105_s9788CXCR4_mouse.pdb
Trajectory/Relaxed/CXCR4_l106_s15325CXCR4_mouse.pdb
Trajectory/Relaxed/CXCR4_l106_s15564CXCR4_mouse.pdb
Trajectory/Relaxed/CXCR4_l106_s22762CXCR4_mouse.pdb
Trajectory/Relaxed/CXCR4_l106_s26386CXCR4_mouse.pdb
Trajectory/Relaxed/CXCR4_l106_s75775CXCR4_mouse.pdb
Trajectory/Relaxed/CXCR4_l107_s24933CXCR4_mouse.pdb
Trajectory/Relaxed/CXCR4_l107_s75675CXCR4_mouse.pdb
Trajectory/Relaxed/CXCR4_l107_s8342CXCR4_mouse.pdb
Trajectory/Relaxed/CXCR4_l107_s84555CXCR4_mouse.pdb
Trajectory/Relaxed/CXCR4_l108_s83259CXCR4_mouse.pdb
Trajectory/Relaxed/CXCR4_l110_s62526CXCR4_mouse.pdb
Trajectory/Relaxed/CXCR4_l110_s83184CXCR4_mouse.pdb
Trajectory/Rel

"\n        u = mda.Universe(file) \n        sel_u = u.select_atoms('chainID A and not type H and not name OXT')\n\n        align.alignto(u, ref, select='chainID A and not type H and not name OXT')\n        sel = u.select_atoms('all')\n        sel.write(f'{outdir}/{design}')\n"

## Detect Clashes

In [58]:
species = ['mouse', 'human']

outdir = 'Trajectory/no_membrane_clashes'
os.makedirs(outdir, exist_ok = True)

for s in species:   
    files = glob.glob(f'Trajectory/Aligned/*_{s}.pdb')
    files.sort()

    for file in files:
        design = os.path.basename(file)

        src_dir = os.path.dirname(file)
        dst_dir = outdir

        u = mda.Universe(file)
        sel = u.select_atoms('chainID B')
        com_z = sel.center_of_mass()[2]
        
        if s == 'human':
            threshold = u.select_atoms('chainID A and name CA and resid 167').center_of_mass()[2]
        else:
            threshold = u.select_atoms('chainID A and name CA and resid 172').center_of_mass()[2]

        if com_z  < threshold:
            pass
        else:
            print(f'probably no membrane clashes: ', design)
            shutil.copy(f'{src_dir}/{design}', f'{dst_dir}/{design}')


probably no membrane clashes:  CXCR4_l100_s87186CXCR4_mouse.pdb
probably no membrane clashes:  CXCR4_l101_s50501CXCR4_mouse.pdb
probably no membrane clashes:  CXCR4_l102_s38852CXCR4_mouse.pdb
probably no membrane clashes:  CXCR4_l103_s82425CXCR4_mouse.pdb
probably no membrane clashes:  CXCR4_l105_s9788CXCR4_mouse.pdb
probably no membrane clashes:  CXCR4_l106_s22762CXCR4_mouse.pdb
probably no membrane clashes:  CXCR4_l106_s26386CXCR4_mouse.pdb
probably no membrane clashes:  CXCR4_l106_s75775CXCR4_mouse.pdb
probably no membrane clashes:  CXCR4_l107_s8342CXCR4_mouse.pdb
probably no membrane clashes:  CXCR4_l107_s84555CXCR4_mouse.pdb
probably no membrane clashes:  CXCR4_l108_s83259CXCR4_mouse.pdb
probably no membrane clashes:  CXCR4_l111_s4194CXCR4_mouse.pdb
probably no membrane clashes:  CXCR4_l113_s53451CXCR4_mouse.pdb
probably no membrane clashes:  CXCR4_l116_s68332CXCR4_mouse.pdb
probably no membrane clashes:  CXCR4_l116_s9124CXCR4_mouse.pdb
probably no membrane clashes:  CXCR4_l117_s1

In [ ]:
# Adjust trajectory CSV
species = ['mouse', 'human']

no_clashes = glob.glob('Trajectory/no_membrane_clashes/*.pdb')
no_clashes = [os.path.basename(i) for i in no_clashes if 'human' in i]
no_clashes = [i.split('CXCR4_human')[0] for i in no_clashes]

for s in species:   
    df = pd.read_csv(f'Trajectory/trajectory_CXCR4_{s}.csv')
    df = df[df["Design"].isin(no_clashes)]
    
    df.to_csv(f'Trajectory/trajectory_CXCR4_{s}_no_clashes.csv', index=False)

# MPNN Relaxed

In [47]:
df = pd.read_csv('output-CXCR4/mpnn_design_stats_hCXCR4.csv')
metrics = [i for i in df.columns if (i.startswith('Average') or i.startswith('Design'))]
df = df[metrics]

# thresholds
with open("settings_filters/default_mulitmer_target_filters.json") as f:
    thresholds = json.load(f)

# drop columns with null threshold
metrics_with_null_threshold = {
    key
    for key, value in thresholds.items()
    if isinstance(value, dict)
    and "threshold" in value
    and value["threshold"] is None
}

cols_to_drop = [col for col in df.columns if col in metrics_with_null_threshold]
df = df.drop(columns=cols_to_drop)
df_filtered = df.sort_values(by='Design')

# summarize failing metrics
fail_counts = {}

for metric, rules in thresholds.items():
    # Skip metrics not in dataframe
    if metric not in df_filtered.columns:
        continue

    threshold = rules.get("threshold")
    higher = rules.get("higher")

    values = df_filtered[metric]

    if higher:
        # FAIL if value < threshold
        fails = (values < threshold).sum()
    else:
        # FAIL if value > threshold
        fails = (values > threshold).sum()

    fail_counts[metric] = fails

fail_df = (
pd.DataFrame.from_dict(fail_counts, orient="index", columns=["n_failed"])
  .sort_values("n_failed", ascending=False)
)
fail_df['percentage'] = fail_df['n_failed'] / len(df_filtered)
fail_df

,n_failed,percentage
Average_n_InterfaceUnsatHbonds,519,0.882653
Average_Binder_RMSD,466,0.792517
Average_Surface_Hydrophobicity,435,0.739796
Average_Binder_pLDDT,387,0.658163
Average_ShapeComplementarity,89,0.151361
Average_n_InterfaceHbonds,49,0.083333
Average_pLDDT,0,0.000000
Average_pTM,0,0.000000
Average_dG,0,0.000000
Average_Binder_Energy_Score,0,0.000000


# AF3 Prediction

In [58]:
TARGET_OPTIONS = {
    "default": "XXXX",
    "hCXCR4": "FREENANFNKIFLPTIYSIIFLTGIVGNGLVILVMGYQKKLRSMTDKYRLHLSVADLLFVITLPFWAVDAVANWYFGNFLCKAVHVIYTVNLYSSVLILAFISLDRYLAIVHATNSQRPRKLLAEKVVYVGVWIPALLLTIPDFIFANVSEADDRYICDRFYPNDLWVVVFQFQHIMVGLILPGIVILSCYCIIISKLSHSKGHQKRKALKTTVILILAFFACWLPYYIGISIDSFILLEIIKQGCEFENTVHKWISITEALAFFHCCLNPILYAFLGAKFKTSAQHALTSVSRGSSLKILSKGKR",
    "mCXCR4": "FRDENVHFNRIFLPTIYFIIFLTGIVGNGLVILVMGYQKKLRSMTDKYRLHLSVADLLFVITLPFWAVDAMADWYFGKFLCKAVHIIYTVNLYSSVLILAFISLDRYLAIVHATNSQRPRKLLAEKAVYVGVWIPALLLTIPDFIFADVSQGDISQGDDRYICDRLYPDSLWMVVFQFQHIMVGLILPGIVILSCYCIIISKLSHSKGHQKRKALKTTVILILAFFACWLPYYVGISIDSFILLGVIKQGCDFESIVHKWISITEALAFFHCCLNPILYAFLGAKFKSSAQHALNSMSRGSSLKILSKGKR"
}

len(TARGET_OPTIONS["mCXCR4"])

311

In [53]:
targets = ['hCXCR4', 'mCXCR4']

for target in targets:
    df = pd.read_csv('output-CXCR4/mpnn_seqs_new.csv')
    df = df.drop(columns=['Design'])
    df = df.rename(columns={'Name':'Design'})
    df['Target'] = target
    df.to_csv(f'output-CXCR4/mpnn_seqs_{target}.csv', index=False)


In [4]:
%%bash
jabba_dir=$( echo /mnt/DATA140To/rita/251227_duo_design_CXCR4/output-CXCR4/AF3_Prediction/ )
target_dir=$( echo /mnt/DATA8To/rita/251227_duo_design_CXCR4/output-CXCR4/AF3_Prediction )

rsync -v -am rita@jabba:$jabba_dir $target_dir

receiving file list ... done

sent 20 bytes  received 7,223,742 bytes  2,063,932.00 bytes/sec
total size is 111,236,211,524  speedup is 15,398.65


# Designs for MD

## Top scoring complexes

In [3]:
topdir = f'/Volumes/LaCie/251227_duo_design_CXCR4/output-CXCR4/AF3_Prediction/output'
df = pd.read_csv(f'{topdir}/scores_CXCR4.csv')[['rank', 'design', 'human_combined_score', 'mouse_combined_score', 'avg_combined_score']]
df
    

,rank,design,human_combined_score,mouse_combined_score,avg_combined_score
0,1,cxcr4_l148_s81911_mpnn6,0.858374,0.839462,0.848918
1,2,cxcr4_l86_s69957_mpnn9,0.845962,0.833619,0.839790
2,3,cxcr4_l109_s23216_mpnn17,0.835576,0.839979,0.837777
3,4,cxcr4_l83_s64403_mpnn17,0.853121,0.816312,0.834716
4,5,cxcr4_l145_s394_mpnn12,0.840806,0.810907,0.825856
5,6,cxcr4_l103_s14597_mpnn4,0.843378,0.798134,0.820756
6,7,cxcr4_l86_s25899_mpnn18,0.826010,0.810852,0.818431
7,8,cxcr4_l124_s63775_mpnn16,0.794041,0.839110,0.816575
8,9,cxcr4_l102_s47749_mpnn8,0.829369,0.802196,0.815782
9,10,cxcr4_l129_s69645_mpnn4,0.827305,0.799935,0.813620


## Repredict binders alone with AF3

In [24]:
def sanitize_filename(name):
    # Allow only alphanumeric characters, spaces, dashes, and underscores.
    safe = "".join(c for c in name if c.isalnum() or c in (' ', '-', '_')).rstrip()
    return safe.replace(" ", "_")

def csv_to_AF3_json(input_csv):
    folder_path = os.path.dirname(os.path.abspath(input_csv))
    
    with open(input_csv, 'r', newline='') as csv_file:
        reader = csv.DictReader(csv_file)
        
        for row in reader:
            design = row.get('Design', '').strip()
            binder = row.get('Sequence', '').strip()
            
            if design and binder:
                json_entry = {
                    "name": design,
                    "modelSeeds": [1],
                    "sequences": [
                        {
                            "protein": {
                                "id": "B",
                                "sequence": binder,
                                "unpairedMsa": "",
                                "pairedMsa": "",
                                "templates": []
                            }
                        }
                    ],
                    "dialect": "alphafold3",
                    "version": 3
                }
                
                safe_design = sanitize_filename(design)
                file_name = f"{safe_design}.json"
                outdir = os.path.join(folder_path, 'AF3_input')
                os.makedirs(outdir, exist_ok=True)
                output_path = os.path.join(outdir, file_name)
                
                with open(output_path, 'w') as json_file:
                    json.dump(json_entry, json_file, indent=2)
                print(f"JSON file saved to {output_path}")
            else:
                print(f"Skipping invalid row: {row}")


In [26]:
topdir = f'/Volumes/LaCie/251227_duo_design_CXCR4/output-CXCR4/AF3_Prediction/output'
df = pd.read_csv(f'{topdir}/scores_CXCR4.csv')[['rank', 'design', 'human_sequence', 'avg_combined_score']]
df = df.rename(columns={'human_sequence' : 'Sequence', 'design':'Design'})
df['Design'] = df.apply(lambda row: f'rank{row["rank"]:02d}_{row["Design"]}', axis=1)

df = df[['Design', 'Sequence']]

topdir = '/Volumes/LaCie/251227_duo_design_CXCR4/output-CXCR4/af3_binder_repredict'
os.makedirs(topdir, exist_ok=True)

df.to_csv(f'{topdir}/AF3_input.csv', index=False)
csv_to_AF3_json(f'{topdir}/AF3_input.csv')


JSON file saved to /Volumes/LaCie/251227_duo_design_CXCR4/output-CXCR4/af3_binder_repredict/AF3_input/rank01_cxcr4_l148_s81911_mpnn6.json
JSON file saved to /Volumes/LaCie/251227_duo_design_CXCR4/output-CXCR4/af3_binder_repredict/AF3_input/rank02_cxcr4_l86_s69957_mpnn9.json
JSON file saved to /Volumes/LaCie/251227_duo_design_CXCR4/output-CXCR4/af3_binder_repredict/AF3_input/rank03_cxcr4_l109_s23216_mpnn17.json
JSON file saved to /Volumes/LaCie/251227_duo_design_CXCR4/output-CXCR4/af3_binder_repredict/AF3_input/rank04_cxcr4_l83_s64403_mpnn17.json
JSON file saved to /Volumes/LaCie/251227_duo_design_CXCR4/output-CXCR4/af3_binder_repredict/AF3_input/rank05_cxcr4_l145_s394_mpnn12.json
JSON file saved to /Volumes/LaCie/251227_duo_design_CXCR4/output-CXCR4/af3_binder_repredict/AF3_input/rank06_cxcr4_l103_s14597_mpnn4.json
JSON file saved to /Volumes/LaCie/251227_duo_design_CXCR4/output-CXCR4/af3_binder_repredict/AF3_input/rank07_cxcr4_l86_s25899_mpnn18.json
JSON file saved to /Volumes/LaCie/2

In [3]:
topdir = f'/Volumes/LaCie/251227_duo_design_CXCR4/output-CXCR4/AF3_Prediction/output'
df = pd.read_csv(f'{topdir}/scores_CXCR4.csv')[['rank', 'design', 'human_sequence', 'avg_combined_score']]

df = df.rename(columns={'human_sequence' : 'Sequence', 'design':'Design'})
df['Design'] = df.apply(lambda row: f'rank{row["rank"]:02d}_{row["Design"]}', axis=1)

df_plddt = pd.read_csv('/Volumes/LaCie/251227_duo_design_CXCR4/output-CXCR4/af3_binder_repredict/scores_CXCR4.csv')[['design', 'ligand_plddt']]
df_plddt = df_plddt.rename(columns={'design':'Design'})

df = pd.merge(df, df_plddt, on='Design')

# filter by ligand plddt
df = df[df['ligand_plddt'] > 0.8]
df

,rank,Design,Sequence,avg_combined_score,ligand_plddt
1,2,rank02_cxcr4_l86_s69957_mpnn9,LHWGSWTLKATFINPKDGSVSSETVTVGPHRIPEEYLNKAKEENYH...,0.839790,0.805408
6,7,rank07_cxcr4_l86_s25899_mpnn18,MVVPKKKKLPAEEVVEQMRWGVSGWMSIYHAEHSLKYMIRSGEIAD...,0.818431,0.817331
8,9,rank09_cxcr4_l102_s47749_mpnn8,SSADPDFPPPLSPLETLALRARLVARMVELLSARLGREPPPYWVHE...,0.815782,0.818608
9,10,rank10_cxcr4_l129_s69645_mpnn4,KVDFKKLSEKEWLIAAMFVYSDPDTPHRINTTEAREKLAKEAGRKL...,0.813620,0.819661
12,13,rank13_cxcr4_l136_s45641_mpnn16,MMTSEELVTKMKEEMLEWFPSEEERKKVEEVFEMLGWEDPSKVKLE...,0.807488,0.855987
18,19,rank19_cxcr4_l97_s5700_mpnn11,KLSFPEYMELHEKIEKELKKDEEFVKEVVGPFLEEEERVWAEVEEK...,0.784947,0.820809
20,21,rank21_cxcr4_l113_s20432_mpnn12,DVKISKIKYIGKTFFIHHNMHYVQYYFEFIDENNKVVKKVSITVYP...,0.776673,0.828184
27,28,rank28_cxcr4_l93_s13262_mpnn11,MKVEIELEDGTIVEIELEEIPEPKNSSVYEFHQYVKELEEKYEEYK...,0.751312,0.805445
28,29,rank29_cxcr4_l121_s3129_mpnn18,HIHHFMTWERYNRHRLEELKRKIEVDGHVLSPIEQERYEQLLRLQE...,0.749826,0.823946
41,42,rank42_cxcr4_l121_s4166_mpnn5,GYMITDMVPWHPSRHHWNYPTTVEGWKEYYLKSYEELKKLVEEGYK...,0.716952,0.814962


In [ ]:
df = pd.read_csv('CXCR4_duo_designs_exp.csv')
df['Design'] = df['Design'].str.replace('_model', '', regex=False)
df = df[['Design', 'human_mean_deltaG', 'mouse_mean_deltaG', 'Sequence', 'binding', 'test']]

# annotate plddt
df_plddt = pd.read_csv('af3_binder_repredict/scores_CXCR4.csv')[['design', 'duo_ligand_plddt']]
df_plddt = df_plddt.rename(columns={'design':'Design'})

df = pd.merge(df, df_plddt, on='Design')
df.to_csv('CXCR4_duo_designs_exp.csv')

,Design,human_mean_deltaG,mouse_mean_deltaG,Sequence,binding,test,duo_ligand_plddt
0,rank20_cxcr4_l113_s46264_mpnn1,-100.000000,-100.000000,GPSKPKHHHHFHGTWIIGWYDLRWQVVDKLLNPEIPDVPMTEEEKK...,both,yes,0.786823
1,rank04_cxcr4_l83_s64403_mpnn17,-100.000000,-86.100035,AEPVELNPWGFPIPNKSLVEKRCKEANVPTPDEFRNEAHKYPTGTP...,both,yes,0.763256
2,rank03_cxcr4_l109_s23216_mpnn17,-100.000000,-64.025291,MVIKYTIQTYYYYDDAGYRTMLTLRIEEVFTDGSSETTVTVSQVWT...,both,yes,0.797793
3,rank14_cxcr4_l141_s54967_mpnn5,-98.070940,-100.000000,PLEGVPEVNGEELNDVEKAVNEAIAAFFEHSRWHRSPPSGNYILFL...,both,yes,0.709825
4,rank22_cxcr4_l87_s59072_mpnn6,-82.622783,-78.042794,MNPMTSHEVWNILKENSEEYDYVNQTGQVVKAYRVAVYNKKDGSLV...,both,yes,0.645572
5,rank06_cxcr4_l103_s14597_mpnn4,-81.885811,-24.688285,MFFWMNVASYDRYMDYRMEREKYENDTTGDPETRRINLVRLGFEYI...,human,no,0.736338
6,rank29_cxcr4_l121_s3129_mpnn18,-69.806697,-25.145834,HIHHFMTWERYNRHRLEELKRKIEVDGHVLSPIEQERYEQLLRLQE...,human,no,0.823946
7,rank05_cxcr4_l145_s394_mpnn12,-69.222225,-28.972545,FHFWTPDHDITRMGPAFIDDPEKYAVGKLNEKDKAQYYASGKKIVD...,human,no,0.791381
8,rank34_cxcr4_l101_s22264_mpnn13,-68.311205,-81.365116,GPWDGLAPREVREQHVLNYAEFILGPGAVPTPEDLFEAVMRMHYWT...,both,yes,0.727355
9,rank15_cxcr4_l90_s70030_mpnn20,-51.887801,-37.382072,WFHDPHTDDGFYIKSMWRFKKYGKNPEDPQNQEVLKLKPKVEEVRE...,human,no,0.639969
